# 04 - 端到端流程执行（可断点续跑）

本 Notebook 用于执行两套流程：

1. baseline 全流程
2. full_best 全流程 — 使用 `configs/experiments/exp_full_best_ddp.yaml`：**关闭** `max_index_rows` / `mcelog_max_rows` 的截断；**`index_features_path` 仍继承 `configs/data.yaml`**。若未改为天池全量索引 CSV，预处理行数仍与演示 stub 相同（并非「自动全量表」）。训练步骤调用 `scripts/train_ddp.sh`（`run_name` 为 `exp_tianchi_full_best_ddp`）。训练日志开头会有 **数据摘要**（索引路径、manifest `n_samples`、train/val 体量）。

并将每一步日志按时间戳保存到 `runs/pipeline-logs/`。

训练/推理/评估的**模型与预测产物**默认写在 `runs/<run_name>/` 下的**时间戳子目录**（如 `20260104_153022_123456/`、`infer_20260104_160530_789012/`），避免重复运行互相覆盖；根目录的 `.latest_train_session` / `.latest_infer_session` 供 `infer`/`evaluate` 自动解析「最近一次」会话（详见 `configs/base.yaml` 中 `versioned_runs`）。

## 与其他 Notebook 的关系

- EDA、预处理特征分析、预测误差分析请使用同目录的 `01_eda.ipynb`、`02_feature_analysis.ipynb`、`03_error_analysis.ipynb`。
- **全目录索引导读**：`notebooks/README.md`（各文件职责、依赖与 `outputs/` 说明）。
- 分析类 Notebook 的图表与导出表写入 `outputs/`（约定见 `outputs/README.md`）；训练/推理主产物仍在 `runs/`。

## 使用建议

- 首次执行：按单元格从上到下运行。
- 中断恢复：重新运行对应实验单元格，已完成步骤会自动跳过。
- 全量重跑：先执行“重置 marker（可选）”单元格，再重新运行流程。
- 若保存过 Notebook 输出后曾改动脚本，建议**重新执行**相关单元，以当前代码为准。
- **训练指标全 nan / 正例率 0**：多为索引与工单打标后 `y` 全负或验证集无正例。仓库内演示索引由 `PYTHONPATH=src python3 scripts/build_demo_index.py` 生成（含正负样本）；换用自建索引后请自行保证标签合理。
- Notebook 内核若曾加载旧版已安装包，请在首格执行后仍异常时 **Restart kernel** 再跑。

## 网格形状与重跑提示

- 切分文件 **`X_*.npy` 形状为 `[N, C, H, W]`**，须与 **`configs/model.yaml`** 的 `in_channels`、`grid_height`、`grid_width` 及 `data/processed/preprocess_manifest.json` 一致。
- 修改过网格或模型输入尺寸、或本地预处理与配置不一致时，请先 **preprocess** 再 **validate**，最后训练。
- 若已清空 `data/processed`、`data/splits` 且尚未 preprocess，可在终端执行 **`bash scripts/validate.sh <实验yaml> --no-require-splits`** 仅校验配置与路径（本 Notebook 内 validate 步骤仍要求已存在切分）。

## 常见错误：切分与 `model.yaml` 的 C/H/W 不一致

多见于 **preprocess 被 marker 跳过**，`data/splits/X_*.npy` 未按新配置重写。

**处理**：删除 `runs/pipeline-logs/markers/` 下该实验的 `<exp>__preprocess.done` 与 `<exp>__validate.done`（或运行文末「重置 marker」），从 **preprocess** 起重跑。建议在 **train 之前**运行下面的「预检」单元格。

> 设计目标：在意外断电/中断后，可以从 marker 继续执行，不必重复跑已完成步骤。

In [1]:
from pathlib import Path
from datetime import datetime

# 从项目根目录或 notebooks/ 子目录运行均可；下面自动解析 PROJECT_ROOT。
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

# 日志目录：保存每一步的时间戳日志
LOG_DIR = PROJECT_ROOT / "runs" / "pipeline-logs"
# marker 目录：保存“该步骤是否已完成”的标记文件
MARKER_DIR = LOG_DIR / "markers"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MARKER_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_DIR:", LOG_DIR)
print("MARKER_DIR:", MARKER_DIR)

PROJECT_ROOT: /root/workspace
LOG_DIR: /root/workspace/runs/pipeline-logs
MARKER_DIR: /root/workspace/runs/pipeline-logs/markers


In [2]:
import subprocess


def run_step(exp_name: str, step_name: str, command: str):
    """执行一个步骤并落盘日志。

    规则：
    - 若 marker 已存在：跳过该步骤（用于断点续跑）
    - 若 marker 不存在：执行命令，保存时间戳日志，成功后写 marker
    """
    marker = MARKER_DIR / f"{exp_name}__{step_name}.done"
    if marker.exists():
        print(f"[SKIP] {exp_name}/{step_name} (marker exists: {marker.name})")
        return

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = LOG_DIR / f"{exp_name}__{step_name}__{ts}.log"
    # set -o pipefail：管道失败则整体非零；PYTHONUNBUFFERED 减少长时间步骤的日志缓冲延迟。
    cmd = f'bash -lc "set -o pipefail; export PYTHONUNBUFFERED=1; {command} 2>&1 | tee \"{log_file}\""'

    print(f"[RUN ] {exp_name}/{step_name}")
    print("      command:", command)
    print("      log:", log_file)
    subprocess.run(cmd, shell=True, check=True, cwd=str(PROJECT_ROOT))
    marker.write_text(ts, encoding="utf-8")
    print(f"[DONE] {exp_name}/{step_name}\n")

In [3]:
# 预检：X_train 的 [C,H,W] 是否与 configs/model.yaml 一致（建议在 train 之前运行）
from pathlib import Path

import numpy as np

try:
    import yaml
except ImportError:
    yaml = None


def preflight_splits_vs_model(root: Path | None = None) -> None:
    root = root or PROJECT_ROOT
    model_p = root / "configs/model.yaml"
    x_train = root / "data/splits/X_train.npy"
    if yaml is None:
        print("[preflight] 跳过：未安装 PyYAML")
        return
    if not model_p.is_file():
        print("[preflight] 跳过：找不到", model_p)
        return
    cfg = yaml.safe_load(model_p.read_text(encoding="utf-8"))
    exp_c = int(cfg.get("in_channels", 3))
    exp_h = int(cfg.get("grid_height", 0))
    exp_w = int(cfg.get("grid_width", 0))
    if not x_train.is_file():
        print("[preflight] 尚无 X_train.npy，请先跑 preprocess")
        return
    x = np.load(x_train, mmap_mode="r")
    if x.ndim != 4:
        print(f"[preflight] 异常：X_train 应为 4 维 [N,C,H,W]，实际 shape={x.shape}")
        return
    _n, c, h, w = x.shape
    c, h, w = int(c), int(h), int(w)
    ok = c == exp_c and h == exp_h and w == exp_w
    if ok:
        print(f"[preflight] 通过：X_train (C,H,W)=({c},{h},{w}) 与 model.yaml 一致")
    else:
        print(
            f"[preflight] 不匹配：X_train (C,H,W)=({c},{h},{w})，"
            f"model.yaml 期望 ({exp_c},{exp_h},{exp_w})。\n"
            "  多半因 marker 跳过了 preprocess。请删除对应 preprocess/validate 的 .done marker 后重跑。"
        )


def preflight_split_labels(root: Path | None = None) -> None:
    root = root or PROJECT_ROOT
    for tag in ("train", "val"):
        p = root / "data" / "splits" / f"y_{tag}.npy"
        if not p.is_file():
            continue
        y = np.load(p)
        pos = int(np.sum(y))
        n = int(len(y))
        if pos == 0:
            print(
                f"[preflight] 警告：{tag} 无正样本 → 验证 AUC/PR 多为 nan，训练易退化为全负预测。"
                " 请检查索引与工单，或运行：`PYTHONPATH=src python3 scripts/build_demo_index.py`"
            )
        else:
            print(f"[preflight] 标签 {tag}: N={n} pos={pos} rate={pos / max(n, 1):.4f}")


preflight_splits_vs_model(PROJECT_ROOT)
preflight_split_labels(PROJECT_ROOT)

[preflight] 不匹配：X_train (C,H,W)=(3,256,256)，model.yaml 期望 (31,128,32)。
  多半因 marker 跳过了 preprocess。请删除对应 preprocess/validate 的 .done marker 后重跑。
[preflight] 标签 train: N=45 pos=29 rate=0.6444
[preflight] 标签 val: N=10 pos=4 rate=0.4000


In [4]:
# baseline 全流程（快速验证）
# 适合先确认环境、脚本、评估产物是否正常。
EXP = "baseline"
CFG = "configs/experiments/exp_baseline.yaml"

run_step(EXP, "preprocess", f"bash scripts/preprocess.sh {CFG}")
run_step(EXP, "validate", f"bash scripts/validate.sh {CFG}")
run_step(EXP, "inspect_data", "bash scripts/inspect_data.sh")
run_step(EXP, "train", f"bash scripts/train.sh {CFG}")
run_step(EXP, "infer", f"bash scripts/infer.sh {CFG}")
run_step(EXP, "evaluate", f"bash scripts/evaluate.sh {CFG}")

[RUN ] baseline/preprocess
      command: bash scripts/preprocess.sh configs/experiments/exp_baseline.yaml
      log: /root/workspace/runs/pipeline-logs/baseline__preprocess__20260512_142722.log
Scanning mcelog for row/col max …
Grid representation='legacy_rc_et', (H,W)=(256,256), channels=3
Loading index CSV …
Loading inventory one-hot …
预处理并行度: worker 预算=67（data.preprocess_workers=null → max(1, floor(os.cpu_count()×preprocess_parallel_frac)) = max(1, floor(112×0.6))；顺序读 mcelog CSV 仍为单线程；Linux 上满足阈值时对「按 sid 排序」「填网格」多进程）
Building mcelog time index（顺序流式读 CSV + 按 sid 聚合；排序/填网格阶段见上并行度）…
提示：distinct sid 数=106（<268），mcelog 按 sid 排序保持单进程。
提示：样本数 n=64 较小（<268），预处理网格填充保持单进程以降低进程开销。
Preprocess done: N=64, pos_rate=0.531250
Preprocess + split completed. [3, 256, 256]
[DONE] baseline/preprocess

[RUN ] baseline/validate
      command: bash scripts/validate.sh configs/experiments/exp_baseline.yaml
      log: /root/workspace/runs/pipeline-logs/baseline__validate__20260512_142725.log
{
  "status": "

In [5]:
# full_best：取消索引/mcelog 行数上限 + DDP（见 yaml 注释）
# 「全量」仅指 max_index_rows / mcelog_max_rows=null；要真全量请改 data.yaml 的 index_features_path 后重跑 preprocess。
# 适合最终结果验证；换全量索引后耗时通常显著高于 baseline。
EXP = "full_best"
CFG = "configs/experiments/exp_full_best_ddp.yaml"

run_step(EXP, "preprocess", f"bash scripts/preprocess.sh {CFG}")
run_step(EXP, "validate", f"bash scripts/validate.sh {CFG}")
run_step(EXP, "inspect_data", "bash scripts/inspect_data.sh")
run_step(EXP, "train", f"bash scripts/train_ddp.sh {CFG}")
run_step(EXP, "infer", f"bash scripts/infer.sh {CFG}")
run_step(EXP, "evaluate", f"bash scripts/evaluate.sh {CFG}")

[RUN ] full_best/preprocess
      command: bash scripts/preprocess.sh configs/experiments/exp_full_best_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_best__preprocess__20260512_142838.log
Scanning mcelog for row/col max …
Grid representation='spatial_heatmap', (H,W)=(128,32), channels=31
Loading index CSV …
Loading inventory one-hot …
预处理并行度: worker 预算=67（data.preprocess_workers=null → max(1, floor(os.cpu_count()×preprocess_parallel_frac)) = max(1, floor(112×0.6))；顺序读 mcelog CSV 仍为单线程；Linux 上满足阈值时对「按 sid 排序」「填网格」多进程）
Building mcelog time index（顺序流式读 CSV + 按 sid 聚合；排序/填网格阶段见上并行度）…
Sorting mcelog buckets by time (fork Pool, 67 tasks, ~67 workers budget) …
提示：样本数 n=64 较小（<268），预处理网格填充保持单进程以降低进程开销。
Preprocess done: N=64, pos_rate=0.515625
Preprocess + split completed. [31, 128, 32]
[DONE] full_best/preprocess

[RUN ] full_best/validate
      command: bash scripts/validate.sh configs/experiments/exp_full_best_ddp.yaml
      log: /root/workspace/runs/pipeline-logs/full_best__va

In [6]:
# 查看日志与 marker（建议每轮执行后查看）
print("\n=== 最近日志文件 ===")
for p in sorted(LOG_DIR.glob("*.log"), key=lambda x: x.stat().st_mtime, reverse=True)[:10]:
    print(p.name)

print("\n=== 已完成 marker ===")
for p in sorted(MARKER_DIR.glob("*.done")):
    print(p.name, "=>", p.read_text(encoding="utf-8").strip())


=== 最近日志文件 ===
full_best__evaluate__20260512_143509.log
full_best__infer__20260512_143506.log
full_best__train__20260512_143410.log
full_best__inspect_data__20260512_143409.log
full_best__validate__20260512_143407.log
full_best__preprocess__20260512_142838.log
baseline__evaluate__20260512_142835.log
baseline__infer__20260512_142832.log
baseline__train__20260512_142728.log
baseline__inspect_data__20260512_142728.log

=== 已完成 marker ===
baseline__evaluate.done => 20260512_142835
baseline__infer.done => 20260512_142832
baseline__inspect_data.done => 20260512_142728
baseline__preprocess.done => 20260512_142722
baseline__train.done => 20260512_142728
baseline__validate.done => 20260512_142725
full_best__evaluate.done => 20260512_143509
full_best__infer.done => 20260512_143506
full_best__inspect_data.done => 20260512_143409
full_best__preprocess.done => 20260512_142838
full_best__train.done => 20260512_143410
full_best__validate.done => 20260512_143407


In [7]:
# 重置 marker（可选）
# 用途：当你希望“从头重跑所有步骤”时执行。
# 风险：删除 marker 后，Notebook 不再跳过已完成步骤，会重新执行对应命令。
removed = 0
for p in sorted(MARKER_DIR.glob("*.done")):
    p.unlink(missing_ok=True)
    removed += 1
print(f"已删除 marker 数量: {removed}")

已删除 marker 数量: 12
